# 04 -- ReportLab et Platypus

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer un PDF bas niveau avec `canvas` ;
- utiliser `SimpleDocTemplate` et les flowables Platypus ;
- ajouter paragraphes, tables, images, espacers ;
- utiliser les styles (`ParagraphStyle`, `StyleSheet`) ;
- construire des documents multi-pages avec header/footer ;
- generer des factures ou rapports structurees ;
- definir des couleurs et polices custom ;
- gerer la pagination.

## Prerequis -- ce que vous devez maitriser

- fonctions typees, pathlib ;
- les bases CSS/typography sont un plus.

Ce que ce notebook **ne couvrira pas** :

- l'extraction (voir notebooks precedents) ;
- WeasyPrint (HTML -> PDF).

## Plan

1. Installation
2. Canvas bas niveau
3. Coordonnees et unites
4. Texte et polices
5. Formes
6. SimpleDocTemplate
7. Paragraphes et styles
8. Tables
9. Styles de tables
10. Images
11. Espacers et Page break
12. Header et footer
13. Facture complete
14. Synthese
15. Exercices

---

## 1. Installation

In [ ]:
# pip install reportlab
import reportlab
reportlab.__version__ if hasattr(reportlab, '__version__') else 'installed'

---

## 2. Canvas bas niveau

In [ ]:
from reportlab.pdfgen import canvas
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir())
path = tmp / 'lowlevel.pdf'

c_pdf = canvas.Canvas(str(path))
c_pdf.drawString(100, 750, 'Hello ReportLab')
c_pdf.save()
print('PDF bas niveau cree')

---

## 3. Coordonnees et unites

Origine en **bas-gauche** (contrairement a Pillow / screen). Unites en points par defaut (1 point = 1/72 pouce).

In [ ]:
from reportlab.lib.units import cm, mm, inch
print('1 cm =', cm, 'points')
print('1 inch =', inch, 'points')

---

## 4. Texte et polices

In [ ]:
c_pdf = canvas.Canvas(str(tmp / 'fonts.pdf'))
c_pdf.setFont('Helvetica-Bold', 20)
c_pdf.drawString(2*cm, 25*cm, 'Titre en gras')
c_pdf.setFont('Helvetica', 12)
c_pdf.drawString(2*cm, 23*cm, 'Texte normal')
c_pdf.save()
print('OK')

### Polices standard

- `Helvetica`, `Helvetica-Bold`, `Helvetica-Oblique`
- `Times-Roman`, `Times-Bold`, `Times-Italic`
- `Courier`, `Courier-Bold`
- `Symbol`, `ZapfDingbats`

---

## 5. Formes

In [ ]:
c_pdf = canvas.Canvas(str(tmp / 'shapes.pdf'))
c_pdf.setStrokeColorRGB(0.8, 0, 0)
c_pdf.setFillColorRGB(1, 1, 0)
c_pdf.rect(2*cm, 20*cm, 5*cm, 3*cm, fill=1, stroke=1)
c_pdf.circle(10*cm, 21.5*cm, 2*cm, fill=1, stroke=1)
c_pdf.line(2*cm, 15*cm, 15*cm, 15*cm)
c_pdf.save()
print('Formes ok')

---

## 6. SimpleDocTemplate

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

styles = getSampleStyleSheet()
doc = SimpleDocTemplate(str(tmp / 'platypus.pdf'), pagesize=A4)
story = [
    Paragraph('<b>Titre</b>', styles['Title']),
    Spacer(1, 0.5*cm),
    Paragraph('Premier paragraphe de contenu.', styles['Normal']),
]
doc.build(story)
print('Platypus OK')

---

## 7. Paragraphes et styles

In [ ]:
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.colors import black, blue, HexColor

title_style = ParagraphStyle(
    name='CustomTitle',
    fontName='Helvetica-Bold',
    fontSize=24,
    textColor=HexColor('#4a90e2'),
    spaceAfter=20,
)
p = Paragraph('Bonjour', title_style)
print('Style custom defini')

---

## 8. Tables

In [ ]:
from reportlab.platypus import Table, TableStyle
from reportlab.lib.colors import grey, whitesmoke, beige, black

data = [
    ['Produit', 'Quantite', 'Prix'],
    ['Pommes', '10', '5.00 EUR'],
    ['Poires', '5', '3.50 EUR'],
    ['Total', '', '8.50 EUR'],
]
table = Table(data, colWidths=[5*cm, 3*cm, 3*cm])
print('Table creee')

---

## 9. Styles de tables

In [ ]:
table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), grey),
    ('TEXTCOLOR', (0, 0), (-1, 0), whitesmoke),
    ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
    ('GRID', (0, 0), (-1, -1), 0.5, black),
    ('BACKGROUND', (0, -1), (-1, -1), beige),
    ('FONTNAME', (0, -1), (-1, -1), 'Helvetica-Bold'),
]))
print('Style applique')

---

## 10. Images

In [ ]:
from reportlab.platypus import Image as PdfImage

# Creer une image test
from PIL import Image as PILImage
img_path = tmp / 'logo.png'
PILImage.new('RGB', (100, 60), 'steelblue').save(img_path)

logo = PdfImage(str(img_path), width=3*cm, height=2*cm)
print('Image ajoutable dans story')

---

## 11. Espacers et Page break

In [ ]:
from reportlab.platypus import PageBreak, Spacer

story2 = [
    Paragraph('Page 1', styles['Title']),
    Spacer(1, 2*cm),
    Paragraph('Contenu page 1', styles['Normal']),
    PageBreak(),
    Paragraph('Page 2', styles['Title']),
]
SimpleDocTemplate(str(tmp / 'multipage.pdf')).build(story2)
print('Multipage')

---

## 12. Header et footer

In [ ]:
def header_footer(canvas_obj, doc_):
    canvas_obj.saveState()
    canvas_obj.setFont('Helvetica', 9)
    canvas_obj.drawString(2*cm, 27.5*cm, 'Mon en-tete')
    canvas_obj.drawRightString(19*cm, 1*cm, f'Page {doc_.page}')
    canvas_obj.restoreState()

doc_hf = SimpleDocTemplate(str(tmp / 'hf.pdf'), pagesize=A4)
doc_hf.build([Paragraph('Contenu', styles['Normal'])], onFirstPage=header_footer, onLaterPages=header_footer)
print('Header/footer OK')

---

## 13. Facture complete

In [ ]:
def generate_invoice(client: str, items: list[tuple[str, int, float]], output: str) -> None:
    doc = SimpleDocTemplate(output, pagesize=A4)
    story = [
        Paragraph('FACTURE', styles['Title']),
        Spacer(1, 1*cm),
        Paragraph(f'<b>Client :</b> {client}', styles['Normal']),
        Spacer(1, 1*cm),
    ]
    data = [['Article', 'Qte', 'Prix U.', 'Total']]
    total = 0.0
    for name, qty, price in items:
        line_total = qty * price
        total += line_total
        data.append([name, str(qty), f'{price:.2f}', f'{line_total:.2f}'])
    data.append(['', '', 'Total', f'{total:.2f}'])
    table = Table(data, colWidths=[7*cm, 2*cm, 3*cm, 3*cm])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), whitesmoke),
        ('GRID', (0, 0), (-1, -1), 0.5, black),
        ('ALIGN', (1, 1), (-1, -1), 'RIGHT'),
        ('BACKGROUND', (0, -1), (-1, -1), beige),
        ('FONTNAME', (0, -1), (-1, -1), 'Helvetica-Bold'),
    ]))
    story.append(table)
    doc.build(story)

generate_invoice('Alice', [('Pommes', 10, 0.5), ('Pain', 3, 2.0)], str(tmp / 'facture.pdf'))
print('Facture generee')

---

## Synthese

| Niveau | Outil |
|---|---|
| Bas | `canvas.Canvas(path)` + `drawString/rect/line` |
| Haut | `SimpleDocTemplate` + flowables Platypus |
| Paragraphe | `Paragraph(text, style)` |
| Table | `Table(data)` + `TableStyle([...])` |
| Image | `Image(path, width, height)` |
| Saut | `PageBreak()`, `Spacer(w, h)` |
| Header/footer | `onFirstPage=`, `onLaterPages=` |

### Regles a retenir

- **Canvas** pour placement precis ; **Platypus** pour flux naturel.
- **Origine bas-gauche** en points (1 pt = 1/72 inch).
- **`doc.build(story)`** execute la mise en page et gere la pagination.
- **`getSampleStyleSheet`** fournit des styles prets a l'emploi.

---

## Exercices

### Exercice 1 (facile) -- Hello PDF

Creez un PDF avec `canvas` contenant le texte 'Bonjour' a 2cm du haut.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.lib.pagesizes import A4

# TODO

<details>
<summary>Voir la correction</summary>

```python
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm

c = canvas.Canvas('bonjour.pdf', pagesize=A4)
w, h = A4
c.setFont('Helvetica', 24)
c.drawString(2*cm, h - 2*cm, 'Bonjour')
c.save()
```

</details>

### Exercice 2 (facile) -- Liste a puces

Avec Platypus, creez un PDF avec un titre 'Courses' et 3 paragraphes (items).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

styles = getSampleStyleSheet()
doc = SimpleDocTemplate('courses.pdf')
story = [Paragraph('<b>Courses</b>', styles['Title']), Spacer(1, 12)]
for item in ['Pommes', 'Pain', 'Lait']:
    story.append(Paragraph(f'- {item}', styles['Normal']))
doc.build(story)
```

</details>

### Exercice 3 (moyen) -- Table stylee

Creez une table de 4 lignes x 3 colonnes (nom, age, ville) avec header colore, grille noire et lignes alternees.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.colors import navy, white, lightgrey, whitesmoke, black

data = [
    ['Nom', 'Age', 'Ville'],
    ['Alice', 30, 'Paris'],
    ['Bob', 25, 'Lyon'],
    ['Eve', 35, 'Marseille'],
]
t = Table(data)
t.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), navy),
    ('TEXTCOLOR', (0, 0), (-1, 0), white),
    ('GRID', (0, 0), (-1, -1), 0.5, black),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [whitesmoke, lightgrey]),
]))
SimpleDocTemplate('table.pdf').build([t])
```

</details>

### Exercice 4 (moyen) -- Multi-pages

Creez un PDF avec une page de garde (titre) puis 3 pages de contenu (PageBreak entre chaque).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

styles = getSampleStyleSheet()
story = [Paragraph('Titre principal', styles['Title']), PageBreak()]
for i in range(1, 4):
    story.append(Paragraph(f'Contenu de la page {i}', styles['Normal']))
    story.append(PageBreak())
SimpleDocTemplate('multi.pdf').build(story)
```

</details>

### Exercice 5 (difficile) -- Rapport avec header/footer et total

Creez une fonction `rapport_ventes(ventes: dict[str, float], output: str)` qui genere un PDF avec :
- header 'Rapport mensuel'
- footer 'Page N'
- une table des ventes
- une ligne de total en fin de table.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.colors import navy, white, black, beige
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm

def rapport_ventes(ventes: dict[str, float], output: str) -> None:
    styles = getSampleStyleSheet()

    def hf(canv, doc_):
        canv.saveState()
        canv.setFont('Helvetica-Bold', 12)
        canv.drawString(2*cm, 28*cm, 'Rapport mensuel')
        canv.setFont('Helvetica', 9)
        canv.drawRightString(19*cm, 1*cm, f'Page {doc_.page}')
        canv.restoreState()

    rows = [['Produit', 'CA']]
    total = 0.0
    for produit, ca in ventes.items():
        rows.append([produit, f'{ca:.2f}'])
        total += ca
    rows.append(['Total', f'{total:.2f}'])

    table = Table(rows, colWidths=[8*cm, 4*cm])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), navy),
        ('TEXTCOLOR', (0, 0), (-1, 0), white),
        ('GRID', (0, 0), (-1, -1), 0.5, black),
        ('BACKGROUND', (0, -1), (-1, -1), beige),
        ('FONTNAME', (0, -1), (-1, -1), 'Helvetica-Bold'),
    ]))
    story = [Paragraph('Ventes du mois', styles['Title']), Spacer(1, 1*cm), table]
    doc = SimpleDocTemplate(output, pagesize=A4)
    doc.build(story, onFirstPage=hf, onLaterPages=hf)
```

</details>

---

## Ressources externes

- [ReportLab User Guide](https://www.reportlab.com/docs/reportlab-userguide.pdf)
- [ReportLab documentation](https://docs.reportlab.com/)